# Trening AI-WAF Content Classifier na Google Colab

Ten notebook odtwarza [ml/train.py](../ml/train.py) z repo `ai-waf-spec` na cudzym (darmowym) CPU/GPU Colaba -- przydatne, gdy Twoja maszyna jest stara jak Palpatine.

**Uwaga:** model to `RandomForestClassifier` na 7 cechach i ~1200 próbkach -- trenuje się w ułamku sekundy nawet na najsłabszym CPU. GPU Colaba i tak nic tu nie przyspieszy (`scikit-learn` go nie używa) -- to ćwiczenie na wygodę (nie trzeba nic instalować lokalnie), nie na wydajność.

Repo jest prywatne, więc klonowanie wymaga Twojego GitHub Personal Access Token (scope `repo`, tylko do odczytu wystarczy). Token wpisujesz sam, poniżej, tylko w tej sesji Colaba -- nigdzie indziej nie jest przesyłany.

In [ ]:
import getpass

GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token (scope: repo): ")
REPO = "pamsmediatech-lang/ai-waf-spec"

!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git /content/ai-waf-spec 2>&1 | tail -5

In [ ]:
%cd /content/ai-waf-spec
# Do samego treningu wystarczy scikit-learn + joblib -- ml/dataset.py
# importuje tylko app/waf/{rules,features,request,session_store}.py,
# żadne z nich nie ciągną fastapi/httpx.
!pip install --quiet scikit-learn joblib

In [ ]:
import sys
sys.path.insert(0, "/content/ai-waf-spec")

from ml.train import train_and_evaluate, save_artifacts

result = train_and_evaluate(n_benign=600, n_malicious=600, seed=42)
metadata = result["metadata"]
print(metadata["report_text"])
print(f"precision={metadata['metrics']['precision']:.4f} recall={metadata['metrics']['recall']:.4f} f1={metadata['metrics']['f1']:.4f}")
print("top features:", metadata["feature_importances"][:3])

In [ ]:
from pathlib import Path

model_path, meta_path = save_artifacts(result["model"], metadata, artifacts_dir=Path("/content/ai-waf-spec/ml/artifacts"))
print(f"saved -> {model_path}")
print(f"saved -> {meta_path}")

from google.colab import files
files.download(str(model_path))
files.download(str(meta_path))

## Opcjonalnie: odeslij wytrenowany artefakt z powrotem do repo

Zamiast ręcznie pobierać pliki (komórka wyżej) i wgrywać je lokalnie,
możesz scommitować je od razu z poziomu Colaba:

In [ ]:
%cd /content/ai-waf-spec
!git config user.email "piotr.szolc@gmail.com"
!git config user.name "Piotr Szolc"
!git add ml/artifacts/
!git commit -m "Retrain content classifier on Colab"
!git push origin master